In [1]:
import json

import numpy as np
import pandas as pd
import photolab as lab
from serving import ImageEncoder

rows = lab.type_photos()
table = pd.DataFrame({"file": [p.name for p, _ in rows], "label": [label for _, label in rows]})
manifest = pd.read_csv(lab.TYPES / "manifest.csv")
assert manifest["file"].is_unique and table["file"].is_unique, "two photos with the same name"
table = table.merge(
    manifest[["file", "query_class", "creator", "licence"]],
    on="file",
    how="left",
    validate="one_to_one",
)
assert table["licence"].isin(["cc0", "pdm"]).all(), "a photo without a public-domain licence"
print(len(table), "photos")
print(table["label"].value_counts().reindex(lab.CLASSES))
moved = table[table["label"] != table["query_class"]]
print(
    f"you moved {len(moved)} of {len(table)} ({len(moved) / len(table):.0%}) "
    "out of the class the search guessed"
)

501 photos
label
people_posing      70
people_working     82
place             124
object             55
vehicle            73
graphic            97
Name: count, dtype: int64
you moved 213 of 501 (43%) out of the class the search guessed


In [3]:
lab.CACHE.mkdir(exist_ok=True)
paths = [p for p, _ in rows]
X = ImageEncoder()(paths)
np.save(lab.CACHE / "types.npy", X)
y = np.array([lab.CLASSES.index(label) for _, label in rows])
print(X.shape, "| every row has length", np.round(np.linalg.norm(X, axis=1)[:3], 3))

(501, 768) | every row has length [1. 1. 1.]


In [5]:
NEAR = 0.86  # looked at the pairs: photos from one shoot scored as low as 0.86
similar = X @ X.T
parent = list(range(len(X)))


def root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


pairs = np.argwhere(np.triu(similar, k=1) > NEAR)
for i, j in pairs:
    parent[root(i)] = root(j)
groups = np.array([root(i) for i in range(len(X))])
print(len(pairs), "near-duplicate pairs;", len(set(groups)), "groups from", len(X), "photos")
# the pairs just under the line: look at a few and decide whether NEAR is right
upper = np.triu(similar, k=1)
close = np.argwhere((upper > NEAR - 0.04) & (upper <= NEAR))[:5]
for i, j in close:
    print(f"  {similar[i, j]:.3f}  {table.file[i]}  {table.file[j]}")

106 near-duplicate pairs; 422 groups from 501 photos
  0.851  ov-0bd644ad.jpg  ov-62edb2e2.jpg
  0.849  ov-14d8948c.jpg  ov-3f1029fb.jpg
  0.828  ov-14d8948c.jpg  ov-f400145b.jpg
  0.823  ov-7237013a.jpg  ov-f2a12feb.jpg
  0.837  ov-cf558d2c.jpg  ov-f2a12feb.jpg


In [6]:
from sklearn.model_selection import StratifiedGroupKFold

outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
fold = np.empty(len(X), dtype=int)
for k, (_, test) in enumerate(outer.split(X, y, groups)):
    fold[test] = k
(lab.HERE / "folds.json").write_text(
    json.dumps(
        {
            "near": NEAR,
            "files": table.file.tolist(),
            "labels": table.label.tolist(),
            "fold": fold.tolist(),
        }
    )
    + "\n"
)
pd.crosstab(table.label, fold).reindex(lab.CLASSES)

col_0,0,1,2,3,4
label,,,,,
people_posing,14,14,14,14,14
people_working,17,17,16,16,16
place,25,25,25,24,25
object,11,11,11,11,11
vehicle,14,14,15,15,15
graphic,20,20,19,19,19
